# Визуализации для регрессии

Графики для анализа качества регрессионной модели.

## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set(style='whitegrid')
%matplotlib inline

## 1. Данные

👉 **Подставь свой датасет и таргет.**

In [ ]:
# ===== ТВОИ НАСТРОЙКИ =====
DATASET_PATH = 'your_dataset.csv'
TARGET_COL   = 'target'
N_TRIALS     = 50          # итерации Optuna
# ===========================

df = pd.read_csv(DATASET_PATH)
df = df.dropna()

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL]).select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Features: {X.shape[1]}, Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

## 2. Обучение моделей + Optuna

In [ ]:
from sklearn.model_selection import cross_val_score

# --- Ridge ---
def obj_ridge(trial):
    a = trial.suggest_float('alpha', 1e-3, 100, log=True)
    return cross_val_score(Ridge(alpha=a), X_train_s, y_train, cv=5, scoring='neg_mean_squared_error').mean()

study_ridge = optuna.create_study(direction='maximize')
study_ridge.optimize(obj_ridge, n_trials=N_TRIALS)
print('Ridge:', study_ridge.best_params)

In [ ]:
# --- Lasso ---
def obj_lasso(trial):
    a = trial.suggest_float('alpha', 1e-4, 100, log=True)
    return cross_val_score(Lasso(alpha=a, max_iter=10000), X_train_s, y_train, cv=5, scoring='neg_mean_squared_error').mean()

study_lasso = optuna.create_study(direction='maximize')
study_lasso.optimize(obj_lasso, n_trials=N_TRIALS)
print('Lasso:', study_lasso.best_params)

In [ ]:
# --- Decision Tree ---
def obj_dt(trial):
    md = trial.suggest_int('max_depth', 2, 30)
    mss = trial.suggest_int('min_samples_split', 2, 20)
    msl = trial.suggest_int('min_samples_leaf', 1, 20)
    return cross_val_score(DecisionTreeRegressor(max_depth=md, min_samples_split=mss,
            min_samples_leaf=msl, random_state=42), X_train, y_train, cv=5,
            scoring='neg_mean_squared_error').mean()

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(obj_dt, n_trials=N_TRIALS)
print('DT:', study_dt.best_params)

In [ ]:
# --- Финальные модели ---
models = {
    'Ridge':  Ridge(**study_ridge.best_params),
    'Lasso':  Lasso(**study_lasso.best_params, max_iter=10000),
    'LR':     LinearRegression(),
    'DT':     DecisionTreeRegressor(**study_dt.best_params, random_state=42),
}

preds = {}
metrics = []

for name, model in models.items():
    if name in ('Ridge', 'Lasso', 'LR'):
        model.fit(X_train_s, y_train)
        yp = model.predict(X_test_s)
    else:
        model.fit(X_train, y_train)
        yp = model.predict(X_test)

    preds[name] = yp
    metrics.append({
        'Model': name,
        'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
        'MAE':  mean_absolute_error(y_test, yp),
        'R2':   r2_score(y_test, yp)
    })

metrics_df = pd.DataFrame(metrics).sort_values('R2', ascending=False)
metrics_df

---
## 3. Actual vs Predicted (все модели)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, yp) in zip(axes, preds.items()):
    ax.scatter(y_test, yp, alpha=0.35, s=14)
    lo = min(y_test.min(), yp.min())
    hi = max(y_test.max(), yp.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=2)
    r2 = r2_score(y_test, yp)
    ax.set_title(f'{name}  (R²={r2:.3f})')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')

plt.tight_layout()
plt.show()

## 4. Residuals vs Predicted (все модели)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, yp) in zip(axes, preds.items()):
    residuals = y_test - yp
    ax.scatter(yp, residuals, alpha=0.35, s=14)
    ax.axhline(0, color='red', linestyle='--', lw=2)
    ax.set_title(f'{name} — Residuals')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 5. Распределение остатков (гистограмма + KDE)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, yp) in zip(axes, preds.items()):
    residuals = y_test - yp
    sns.histplot(residuals, kde=True, ax=ax, edgecolor='black')
    ax.axvline(0, color='red', linestyle='--', lw=2)
    ax.set_title(f'{name} — Residual distribution')
    ax.set_xlabel('Residual')

plt.tight_layout()
plt.show()

## 6. Q-Q plot остатков (нормальность)

In [ ]:
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, yp) in zip(axes, preds.items()):
    residuals = (y_test - yp).values
    stats.probplot(residuals, dist='norm', plot=ax)
    ax.set_title(f'{name} — Q-Q plot')
    ax.get_lines()[0].set_markersize(4)

plt.tight_layout()
plt.show()

## 7. Сравнение метрик — bar charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    bars = ax.bar(metrics_df['Model'], metrics_df[metric], edgecolor='black')
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Коэффициенты линейных моделей

In [ ]:
coef_df = pd.DataFrame({'Feature': X.columns})
coef_df['Ridge'] = models['Ridge'].coef_
coef_df['Lasso'] = models['Lasso'].coef_
coef_df['LR']    = models['LR'].coef_
coef_df = coef_df.set_index('Feature')

coef_df.plot.bar(figsize=(max(8, len(X.columns) * 0.6), 5), edgecolor='black')
plt.title('Коэффициенты линейных моделей')
plt.xticks(rotation=45, ha='right')
plt.axhline(0, color='black', lw=0.5)
plt.tight_layout()
plt.show()

## 9. Feature Importances — Decision Tree

In [ ]:
imp = pd.Series(models['DT'].feature_importances_, index=X.columns)
imp = imp[imp > 0].sort_values(ascending=True)

if len(imp) == 0:
    print('Все feature importances = 0 (дерево может быть слишком мелким).')
else:
    plt.figure(figsize=(6, max(3, len(imp) * 0.35)))
    imp.plot.barh(edgecolor='black', color='steelblue')
    plt.title('Decision Tree — Feature Importances')
    plt.tight_layout()
    plt.show()

## 10. Predicted vs Actual — все модели на одном графике

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

colors = {'Ridge': 'tab:blue', 'Lasso': 'tab:orange', 'LR': 'tab:green', 'DT': 'tab:red'}

for name, yp in preds.items():
    ax.scatter(y_test, yp, alpha=0.3, s=14, label=name, color=colors[name])

lo = min(y_test.min(), min(p.min() for p in preds.values()))
hi = max(y_test.max(), max(p.max() for p in preds.values()))
ax.plot([lo, hi], [lo, hi], 'k--', lw=2, label='Ideal')

ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('Все модели — Actual vs Predicted')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 11. Распределение ошибок — все модели на одном графике

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name, yp in preds.items():
    residuals = y_test - yp
    sns.kdeplot(residuals, label=name, ax=ax, linewidth=2)

ax.axvline(0, color='black', linestyle='--', lw=1)
ax.set_title('Распределение остатков — все модели')
ax.set_xlabel('Residual')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 12. Optuna — важность гиперпараметров

In [ ]:
from optuna.importance import get_param_importances

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, study) in zip(axes, [('Ridge', study_ridge),
                                     ('Lasso', study_lasso),
                                     ('DT', study_dt)]):
    imp = get_param_importances(study)
    if imp:
        pd.Series(imp).plot.barh(ax=ax, edgecolor='black', color='steelblue')
        ax.set_title(f'{name} — Param importance')
        ax.set_xlabel('Importance')
    else:
        ax.text(0.5, 0.5, 'Not enough trials', ha='center', va='center',
                transform=ax.transAxes)
        ax.set_title(name)

plt.tight_layout()
plt.show()

## 13. Optuna — история оптимизации

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, study) in zip(axes, [('Ridge', study_ridge),
                                     ('Lasso', study_lasso),
                                     ('DT', study_dt)]):
    trials = study.trials
    values = [t.value for t in trials if t.value is not None]
    ax.plot(range(len(values)), values, '-o', markersize=3)
    ax.set_title(f'{name} — CV score')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Neg MSE')

plt.tight_layout()
plt.show()